# Model językowy tylko-dekoder - od zeraImplementacja autoregresyjnego modelu językowego (GPT) w czystym PyTorchu: maska przyczynowa, wielogłowicowa atencja, blok Transformera, pretrening na korpusie Wolnych Lektur i strategie generowania tekstu. Na końcu - model pretrenowany (`phi-1_5`) wczytany w kwantyzacji 8-bitowej.

In [ ]:
!pip install -q transformers datasets torchmetrics gdown

## Tokenizator`sdadas/polish-gpt2-small` - tokenizator BPE dla języka polskiego.

In [ ]:
from transformers import AutoTokenizer

ckpt = "sdadas/polish-gpt2-small"

tokenizer = AutoTokenizer.from_pretrained(ckpt)
tokenizer.pad_token = tokenizer.eos_token

sample_text = "This is a sample text"

tokens = tokenizer.tokenize(sample_text)
intput_ids = tokenizer.convert_tokens_to_ids(tokens)
print(tokens)
print(intput_ids)
tokens_decoded = tokenizer.convert_ids_to_tokens(intput_ids)
print(tokens_decoded)

print("-" * 10)

input_ids = tokenizer.encode(sample_text)
text_reconstructed = tokenizer.decode(input_ids)
print(input_ids)
print(text_reconstructed)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/835 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/319 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/72.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

['Th', 'is', 'Ġis', 'Ġa', 'Ġsam', 'ple', 'Ġte', 'xt']
[13653, 1344, 6975, 330, 509, 5967, 348, 23895]
['Th', 'is', 'Ġis', 'Ġa', 'Ġsam', 'ple', 'Ġte', 'xt']
----------
[13653, 1344, 6975, 330, 509, 5967, 348, 23895]
This is a sample text


In [ ]:
print("Metoda __call__ przygotowuje cały wsad treningowy")

inputs = ["Ale ma kota", "Kot jest biały i ma Alę"]
tokenized_inputs = tokenizer(inputs, return_tensors="pt", padding=True)
print(tokenized_inputs['input_ids'])
print(tokenized_inputs['attention_mask'])

Metoda __call__ przygotowuje cały wsad treningowy
tensor([[  870,   438, 14331,     2,     2,     2,     2],
        [33540,   377, 10769,   292,   438,   456,   873]])
tensor([[1, 1, 1, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1]])


In [ ]:
print(f"{tokenizer.vocab_size=}")
list(tokenizer.vocab.items())[2]

tokenizer.vocab_size=51200


('Ġspowi', 25910)

## Konfiguracja modeluModel autoregresyjny: ta sama operacja wywoływana sekwencyjnie, każdorazowo z uwzględnieniem poprzedniego wyniku. Uczenie samonadzorowane - etykietą jest następny token w sekwencji.Kontekst 256 tokenów, 12 warstw, 12 głowic, wymiar embeddingu 768.

In [ ]:
# Model **autoregresyjny** - sekwencyjne wywoływanie tej samej opracacji,
# biorąc pod uwagę wyniki poprzedniej
# Uczenie samonadzorowane - algorytm automatycznie ekstrahuje etykiety do przewidywania
# (następny token w sekwencji)

# PARAMETRY MODELU
# 1. Rozmiar kontekstu 256 tokenów
# 2. Rozmiar słownika 51200 tokenów
LLM_CONFIG = {
    "vocab_size": tokenizer.vocab_size,
    "context_length": 256,
    "emb_dim": 768,
    "n_layers": 12,
    "n_heads": 12,
    "feedforward_dim_mult": 4,
    "add_kv_bias": False,
    "dropout_rate": 0.1
}


In [ ]:
from torch import nn

def print_param_count(model: nn.Module):
  total_size_bytes = sum(p.numel() * p.element_size() for p in model.parameters())
  total_size_mb = total_size_bytes / (1024 * 1024)

  total_params = sum(p.numel() for p in model.parameters())
  trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

  print(f'Number of parameters - Total: {total_params:,d} ({total_size_mb:.1f} MB)   Trainable: {trainable_params:,d}')

### Maska przyczynowa i atencjaMaska górnotrójkątna z `-inf` nad przekątną - token nie może patrzeć w przyszłość.

In [ ]:
import torch

def get_attention_mask(seq_size):
    return torch.triu(torch.full((seq_size, seq_size), float("-inf")), diagonal=1)

print(get_attention_mask(4))

tensor([[0., -inf, -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., 0., -inf],
        [0., 0., 0., 0.]])


In [ ]:
att = nn.MultiheadAttention(
    embed_dim=LLM_CONFIG["emb_dim"],
    num_heads=LLM_CONFIG["n_heads"],
    dropout=LLM_CONFIG["dropout_rate"],
    add_bias_kv=LLM_CONFIG["add_kv_bias"],
    batch_first=True
)
print(att)
print(print_param_count(att))

MultiheadAttention(
  (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
)
Number of parameters - Total: 2,362,368 (9.0 MB)   Trainable: 2,362,368
None


## ArchitekturaBloki budowane ręcznie: warstwa w pełni połączona z rozszerzeniem 4×, blok Transformera (pre-norm: LayerNorm → atencja → residual → LayerNorm → FFN → residual), a na wierzchu model językowy z embeddingami tokenów i pozycji oraz głowicą rzutującą na słownik.

In [ ]:
x = torch.rand(1, 4, LLM_CONFIG["emb_dim"])
att_output, att_weights = att(x, x, x, attn_mask=get_attention_mask(4))

print(f"{att_output.shape=} {att_weights.shape=}")
print(f"{att_weights=}")

att_output.shape=torch.Size([1, 4, 768]) att_weights.shape=torch.Size([1, 4, 4])
att_weights=tensor([[[1.0185, 0.0000, 0.0000, 0.0000],
         [0.5310, 0.4875, 0.0000, 0.0000],
         [0.3254, 0.3241, 0.3074, 0.0000],
         [0.2659, 0.2480, 0.2739, 0.2113]]], grad_fn=<MeanBackward1>)


In [ ]:
class FeedForwardLayer(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    hidden_size = cfg["feedforward_dim_mult"] * cfg["emb_dim"]

    self.layers = nn.Sequential(
        nn.Linear(cfg["emb_dim"], hidden_size),
        nn.GELU(),
        nn.Linear(hidden_size, cfg["emb_dim"])
    )

  def forward(self, X):
    return self.layers(X)


In [ ]:
class TransformerBlock(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.att = nn.MultiheadAttention(
        embed_dim=cfg["emb_dim"],
        num_heads=cfg["n_heads"],
        dropout=cfg["dropout_rate"],
        add_bias_kv=cfg["add_kv_bias"],
        batch_first=True
    )
    self.layer_norm_1 = nn.LayerNorm(cfg["emb_dim"])
    self.feed_forward = FeedForwardLayer(cfg)
    self.layer_norm_2 = nn.LayerNorm(cfg["emb_dim"])
    self.dropout = nn.Dropout(cfg["dropout_rate"])

  def forward(self, x, padding_mask=None):
    seq_len = x.shape[1]
    attn_mask = torch.triu(torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool), diagonal=1)

    res = x
    x = self.layer_norm_1(x)
    x = self.att(
        x, x, x,
        is_causal=True,
        need_weights=False,
        attn_mask=attn_mask,
        key_padding_mask=padding_mask
    )[0]
    x = self.dropout(x)
    x = x + res

    res = x
    x = self.layer_norm_2(x)
    x = self.feed_forward(x)
    x = self.dropout(x)
    x = x + res

    return x

In [ ]:
transformer_block = TransformerBlock(LLM_CONFIG)
print_param_count(transformer_block)

Number of parameters - Total: 7,087,872 (27.0 MB)   Trainable: 7,087,872


In [ ]:
class LlmModel(nn.Module):
  def __init__(self, cfg):
    super().__init__()
    self.vocab_size = cfg["vocab_size"]
    self.emb_dim = cfg["emb_dim"]
    self.context_length = cfg["context_length"]

    self.emb_layer = nn.Embedding(self.vocab_size, self.emb_dim)
    self.pos_emb_layer = nn.Embedding(self.context_length, self.emb_dim)
    self.drop_emb = nn.Dropout(cfg["dropout_rate"])

    self.transformer_blocks = nn.ModuleList([
        TransformerBlock(cfg) for _ in range(cfg["n_layers"])
    ])

    self.classification_norm = nn.LayerNorm(cfg["emb_dim"])
    self.classification_layer = nn.Linear(cfg["emb_dim"], cfg["vocab_size"])

  def forward(self, x, padding_mask=None):
    """Predict next token
    Params
    :X: (batch_size, seq_len, 1) tensor of token ids
    """
    batch_size, seq_len = x.shape

    pos = torch.arange(seq_len, dtype=torch.long, device=x.device)
    pos_embeddings = self.pos_emb_layer(pos)
    embeddings = self.emb_layer(x)
    x = self.drop_emb(embeddings + pos_embeddings)

    if padding_mask is not None:
      padding_mask = (padding_mask == 0)

    for transformer_block in self.transformer_blocks:
      x = transformer_block(x, padding_mask)

    logits = self.classification_layer(self.classification_norm(x))
    return logits

In [ ]:
llm_model = LlmModel(LLM_CONFIG)
print_param_count(llm_model)

Number of parameters - Total: 163,947,008 (625.4 MB)   Trainable: 163,947,008


In [ ]:
import torch

sample_text = ["Hej, jestem"]
tokenized_inputs = tokenizer(sample_text, return_tensors="pt", padding=True)
print(f"{tokenized_inputs['input_ids']=}")
print(f"{tokenized_inputs['attention_mask']=}")

llm_model.eval()

logits = llm_model(
    tokenized_inputs["input_ids"],
    tokenized_inputs["attention_mask"]
)

token_ids = torch.argmax(logits, dim=-1)
tokens = tokenizer.convert_ids_to_tokens(token_ids[0])
decoded_text = tokenizer.batch_decode(token_ids)

print(f"{tokens=}")
print(f"Model generated: {decoded_text}")

tokenized_inputs['input_ids']=tensor([[17641,    16,  1382]])
tokenized_inputs['attention_mask']=tensor([[1, 1, 1]])
tokens=['glÄħdaÅĤ', 'Ġklasycznych', 'Ġmg']
Model generated: ['glądał klasycznych mg']


## Zbiór danych - Wolne LekturyTeksty dzielone na nachodzące na siebie fragmenty (stride = połowa kontekstu), tokenizowane partiami.

In [ ]:
!gdown 1aQ1AK9X07saOCIjmirAlcRbJPQnjv7GG

Downloading...
From (original): https://drive.google.com/uc?id=1aQ1AK9X07saOCIjmirAlcRbJPQnjv7GG
From (redirected): https://drive.google.com/uc?id=1aQ1AK9X07saOCIjmirAlcRbJPQnjv7GG&confirm=t&uuid=ed22184b-1f11-4339-9a41-f4fd891d5172
To: /content/wolne_lektury_dataset_dump.pkl
100% 284M/284M [00:01<00:00, 172MB/s]


In [ ]:
import pickle

with open('wolne_lektury_dataset_dump.pkl', 'rb') as f:
    dataset_dump = pickle.load(f)

In [ ]:
from typing import no_type_check_decorator

def print_dataset_stats(ds):
  print(f"Liczba dokumentów w zbiorze danych: {len(ds):,}")
  n_char = 0
  for doc in ds:
    n_char += len(doc)
  print(f"Liczba znaków w zbiorze danych: {n_char:,}")
  print(f"Średnia długość dokumentu: {n_char/len(ds):.1f}")
  print(f"Maksymalna długość dokumentu: {max([len(doc) for doc in ds]):,}")
  print(f"Minimalna długość dokumentu: {min([len(doc) for doc in ds]):,}")

print_dataset_stats(dataset_dump)

Liczba dokumentów w zbiorze danych: 6,619
Liczba znaków w zbiorze danych: 262,196,111
Średnia długość dokumentu: 39612.6
Maksymalna długość dokumentu: 2,438,186
Minimalna długość dokumentu: 35


In [ ]:
from tqdm.auto import tqdm
import bisect

class WolneLekturyDataset(torch.utils.data.Dataset):
  def __init__(self, dataset_dump, tokenizer, chunk_size, stride, batch_tokenize_size):
    self.chunk_size = chunk_size
    self.stride = stride
    self.total_chunks = 0
    self.tokens = []
    self.cumulative_chunks = []

    count_skipped = 0
    min_required_len = chunk_size + 1

    for doc_index in tqdm(range(0, len(dataset_dump), batch_tokenize_size)):
      tokenized_docs = tokenizer(
          dataset_dump[doc_index: doc_index + batch_tokenize_size],
          return_attention_mask=False,
          add_special_tokens=False,
          padding=False
      )

      for tokenized_doc in tokenized_docs["input_ids"]:
        chunks_in_doc = (len(tokenized_doc) - min_required_len) // stride + 1
        self.tokens.append(torch.tensor(tokenized_doc, dtype=torch.long))
        self.total_chunks += chunks_in_doc
        self.cumulative_chunks.append(self.total_chunks)

  def __len__(self):
    return self.total_chunks

  def __getitem__(self, idx):
    doc_idx = bisect.bisect_right(self.cumulative_chunks, idx)

    if doc_idx == 0:
        chunk_in_docs_indx = idx
    else:
        chunk_in_docs_indx = idx - self.cumulative_chunks[doc_idx - 1]

    start_index = chunk_in_docs_indx * self.stride
    doc_tensor = self.tokens[doc_idx]

    item = {
        'input_ids': doc_tensor[start_index : start_index + self.chunk_size],
        'labels': doc_tensor[start_index + 1 : start_index + self.chunk_size + 1]
    }
    return item


In [ ]:
train_size = 200
val_size = 10
chunk_size = LLM_CONFIG["context_length"]
stride = chunk_size // 2
batch_tokenize_size = 10

train_dataset = WolneLekturyDataset(dataset_dump[:train_size], tokenizer, chunk_size, stride, batch_tokenize_size)
val_dataset = WolneLekturyDataset(dataset_dump[-val_size:], tokenizer, chunk_size, stride, batch_tokenize_size)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)

  0%|          | 0/20 [00:00<?, ?it/s]

  0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
! pip install torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 23.9 MB/s eta 0:00:00


## PretreningAdamW z akumulacją gradientu (4 kroki), `CosineAnnealingLR`, metryki liczone przez `torchmetrics`, checkpointy co 1/10 treningu.

In [ ]:
import torchmetrics
from tqdm.auto import tqdm
import torch

def train_model(model, train_loader, val_loader, epochs, device, checkpoint_path=None):
  model.to(device)

  accumulation_steps = 4
  steps_per_epoch = (len(train_loader) + accumulation_steps - 1) // accumulation_steps
  total_steps = epochs * steps_per_epoch
  checkpoint_steps_freq = total_steps // 10

  optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
  lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-5)
  criterion = torch.nn.CrossEntropyLoss()

  metric_loss = torchmetrics.MeanMetric().to(device)
  metric_acc = torchmetrics.classification.Accuracy(task="multiclass", num_classes=LLM_CONFIG["vocab_size"]).to(device)

  total_step = 0
  for epoch in range(epochs):
    model.train()

    # TRAINING
    train_pbar = tqdm(total=steps_per_epoch)
    train_pbar.set_description(f"Epoch {epoch}")

    optimizer.zero_grad()

    if checkpoint_path is not None:
      resume_step = int(checkpoint_path.split("_")[-1].split(".")[0])
      total_step = resume_step
      train_pbar.update(resume_step)
      train_pbar.refresh()
      model.load_state_dict(torch.load(checkpoint_path))
      print(f"Loaded checkpoint from step {resume_step}")

    for batch_idx, batch in enumerate(train_loader):
      input_ids = batch["input_ids"].to(device)

      logits = model(input_ids)
      logits_flat = logits.view(-1, LLM_CONFIG["vocab_size"])
      labels_flat = batch["labels"].to(device).view(-1)

      loss = criterion(logits_flat, labels_flat)
      scaled_loss = loss / accumulation_steps
      scaled_loss.backward()

      if (batch_idx + 1) % accumulation_steps == 0 or (batch_idx + 1) == len(train_loader):
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

        train_pbar.update(1)

        if total_step % checkpoint_steps_freq == 0:
          torch.save(model.state_dict(), f"model_step_{total_step}.pt")

        total_step += 1

      metric_loss(loss)
      _, preds = torch.max(logits_flat, dim=-1)
      metric_acc(preds, labels_flat)

      current_lr = optimizer.param_groups[0]['lr']

      train_pbar.set_postfix({
          'loss': f"{loss.item():.4f}",
          'lr': f"{current_lr:.2e}"
      })

    train_pbar.close()

    train_loss = metric_loss.compute()
    train_acc = metric_acc.compute()
    metric_loss.reset()
    metric_acc.reset()

    # VALIDATION
    model.eval()
    with torch.no_grad():
      for batch in val_loader:
        input_ids = batch["input_ids"].to(device)

        logits = model(input_ids)

        logits_flat = logits.view(-1, LLM_CONFIG["vocab_size"])
        labels_flat = batch["labels"].to(device).view(-1)
        loss = criterion(logits_flat, labels_flat)
        metric_loss(loss)

        _, preds = torch.max(logits_flat, dim=-1)
        metric_acc(preds, labels_flat)

    val_loss = metric_loss.compute()
    val_acc = metric_acc.compute()
    metric_loss.reset()
    metric_acc.reset()

    print(f"Train loss: {train_loss:.4f} | Train acc: {train_acc:.4f} | Val acc: {val_acc:.4f}")

Epoch 0:   0%|          | 1/3928 [05:30<360:17:14, 330.29s/it, loss=8.4241, lr=1.00e-04]


In [ ]:
print_param_count(llm_model)

Number of parameters - Total: 163,947,008 (625.4 MB)   Trainable: 163,947,008


In [ ]:
epochs = 3
device = "cuda" if torch.cuda.is_available() else "cpu"
train_model(llm_model, train_loader, val_loader, epochs, device)

  0%|          | 0/545 [00:00<?, ?it/s]

Train loss: 6.7857 | Train acc: 0.1513 | Val acc: 0.2021


  0%|          | 0/545 [00:00<?, ?it/s]

Train loss: 6.1311 | Train acc: 0.1805 | Val acc: 0.2112


  0%|          | 0/545 [00:00<?, ?it/s]

In [ ]:
torch.save(llm_model.state_dict(), "model.pt")

In [ ]:
print(next(iter(train_loader)))

In [ ]:
prompts = ["Hej, jestem", "Twoja mamaj jest", "Bartek to"]
k = 3

llm_model.eval()

for prompt in prompts:
  tokenized_prompt = tokenizer(prompt, return_tensors="pt", padding=True)

  with torch.no_grad():
    logits = llm_model(
        tokenized_prompt["input_ids"],
        tokenized_prompt["attention_mask"]
    )
    next_token_pred_logits = logits[0, -1, :]
    probas, pos = torch.topk(next_token_pred_logits, k)

    results = {tokenizer.decode(ndx): pp.item() for (ndx, pp) in zip(pos, probas)}

    print(f"Prompt: {prompt}")
    print(f"Model predicted: {results}")

## Strategie generowania tekstuGenerowanie autoregresyjne z podmienianą funkcją próbkowania - zachłanne (`argmax`) oraz z temperaturą.

In [ ]:
def generate_text(model, tokenizer, prompt, max_tokens, device, sampling_func):
  model.eval()

  context_size = model.context_length

  tokenized_prompt = tokenizer(prompt, return_tensors="pt", padding=True).to(device)
  input_ids = tokenized_prompt["input_ids"]

  eos_token_id = getattr(tokenizer, "eos_token_id", None)

  with torch.no_grad():
    for _ in range(max_tokens):
      logits = model(input_ids[:, -context_size:])
      next_token_pred_logits = logits[:, -1, :]

      next_token_id = sampling_func(next_token_pred_logits)

      input_ids = torch.cat([input_ids, next_token_id.unsqueeze(0)], dim=-1)

      if eos_token_id is not None and next_token_id.item() == eos_token_id:
        break

  generated_text = tokenizer.batch_decode(input_ids)
  return generated_text

In [ ]:
def greedy_sampling(next_token_pred_logits):
  next_token_id = torch.argmax(next_token_pred_logits, dim=-1, keepdim=True)
  return next_token_id

def top_k_sampling(next_token_pred_logits, k):
  top_k_values, top_k_indices = torch.topk(next_token_pred_logits, k, dim=-1)
  probs = torch.softmax(top_k_values, dim=-1)

  sampled_indexes = torch.multinomial(probs, num_samples=1)
  next_token_id = top_k_indices.gather(top_k_indices, dim=-1, index=sampled_indexes)

  return next_token_id

def temperature_sampling(next_token_pred_logits, temperature):
  scaled_logits = next_token_pred_logits / temperature
  probs = torch.softmax(scaled_logits, dim=-1)

  next_token_id = torch.multinomial(probs, num_samples=1)

  return next_token_id

In [ ]:
val_batch = iter(val_loader)
n_prompts = 3
prompts = [next(val_batch)[0]["text"] for _ in range(n_prompts)]

generated_text = generate_text(
    llm_model,
    tokenizer,
    prompts,
    max_tokens=100,
    device=device,
    sampling_func=greedy_sampling
)
print(generated_text)

## Model pretrenowany w kwantyzacji 8-bitowej`microsoft/phi-1_5` wczytany przez `bitsandbytes` w 8 bitach - porównanie zajętości pamięci i generowanie z próbkowaniem temperaturowym.

In [ ]:
!pip install -q -U transformers
!pip install -q -U datasets
!pip install -q -U peft
!pip install -q -U bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.5 MB/s eta 0:00:00


In [ ]:
import datasets
import peft
import transformers
import bitsandbytes # do kwantyzacji

In [ ]:
import numpy as np
import torch
from torch import nn
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForLanguageModeling, BitsAndBytesConfig

In [ ]:
q_config = BitsAndBytesConfig(load_in_8bit=True)

base_model_id = "microsoft/phi-1_5"

model = AutoModelForCausalLM.from_pretrained(
    base_model_id, quantization_config=q_config, dtype=torch.float16, trust_remote_code=True, low_cpu_mem_usage=True
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/736 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.84G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

In [ ]:
print(model)

PhiForCausalLM(
  (model): PhiModel(
    (embed_tokens): Embedding(51200, 2048)
    (layers): ModuleList(
      (0-23): 24 x PhiDecoderLayer(
        (self_attn): PhiAttention(
          (q_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (v_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
          (dense): Linear8bitLt(in_features=2048, out_features=2048, bias=True)
        )
        (mlp): PhiMLP(
          (activation_fn): NewGELUActivation()
          (fc1): Linear8bitLt(in_features=2048, out_features=8192, bias=True)
          (fc2): Linear8bitLt(in_features=8192, out_features=2048, bias=True)
        )
        (input_layernorm): LayerNorm((2048,), eps=1e-05, elementwise_affine=True)
        (resid_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (rotary_emb): PhiRotaryEmbedding()
    (embed_dropout): Dropout(p=0.0, inplace=False)
    (final_la

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
print(tokenizer.vocab_size)

50257


In [ ]:
print(f"Zajętość pamięci przez model: {model.get_memory_footprint() / 1024**3:0.3f}GB")

Zajętość pamięci przez model: 1.517GB


In [ ]:
text = "This is a test message"
tokenized = tokenizer(text)
print(tokenized)
decoded = tokenizer.decode(tokenized["input_ids"])
print(decoded)

{'input_ids': [1212, 318, 257, 1332, 3275], 'attention_mask': [1, 1, 1, 1, 1]}
This is a test message


In [ ]:
def temperature_sampling(next_token_logits, temperature=1.5):
  scaled_logits = next_token_logits / temperature
  probas = torch.softmax(scaled_logits, dim=-1)
  next_token_id = torch.multinomial(probas, num_samples=1)
  return next_token_id

def generate_text_from_prompt_batch(model, tokenizer, prompts_batch, max_tokens, device, sampling_func):
  model.eval()
  tokenized_batch = tokenizer(prompts_batch, padding=True)
  input_ids = tokenized_batch["input_ids"]
  mask = tokenized_batch["attention_mask"]

  eos_token_id = getattr(tokenizer, "eos_token_id", None)

  for i in range(max_tokens):
    with torch.no_grad():
      logits = model(**tokenized_batch)

    next_token_logits = logits[:, -1, :]
    next_token_id = sampling_func(next_token_logits)
    tokenized_batch["input_ids"] = torch.cat([tokenized_batch["input_ids"], next_token_id], dim=-1)

    if eos_token_id is not None and next_token_id.item() == eos_token_id:
      break

  generated_text = tokenizer.batch_decode(tokenized_batch["input_ids"])
  return generated_text

In [ ]:
from colorama import Fore
from transformers import BatchEncoding

device = 'cuda'


def generate_text_pretrained_model(model: nn.Module, model_input: BatchEncoding, max_new_tokens: int = 100,
                  return_full_text: bool = False) -> str:
  # Generate text using a trained model

  model.eval()
  with torch.no_grad():
    generated_tokens = model.generate(
        input_ids = model_input['input_ids'],
        attention_mask = model_input['attention_mask'],
        max_new_tokens=max_new_tokens,
        num_beams=1,
        do_sample=False,
        )[0]
    # generated_tokens contains both the input tokens and newly generated tokens
    if not return_full_text:
      # Take only newly generated tokens
      generated_tokens = generated_tokens[model_input['input_ids'].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)


def generate_and_print_text(model: nn.Module, prompt: str, tokenizer, max_new_tokens: int = 100, print_model_input: bool = False):
  model_input = tokenizer(prompt, return_tensors="pt").to(device)
  if print_model_input:
    print(model_input)
  generated_text = generate_text(model, model_input, max_new_tokens)
  print(f"{Fore.BLACK}{prompt}", end="")
  print(f"{Fore.BLUE}{generated_text}")